# 1-Minute → Any Timeframe Aggregation (Nautilus Internal + Custom)

Generalizes the custom 1-min aggregator so the **target timeframe is a runtime parameter**.
Bucket-close logic handles any unit — minutes, hours, days, weeks, months, years — with
arbitrary multipliers (`15min`, `45min`, `2h`, `1d`, `2w`, `1mo`, `2mo`, `1y`, ...).

| | Strategy | Subscribes to | Aggregation done by | Output |
|-|----------|---------------|---------------------|--------|
| 1 | `NautilusInternalStrategy` | `<N>-<UNIT>-…-INTERNAL@1-MINUTE-EXTERNAL` | Nautilus DataEngine (built-in) | OHLCV CSV |
| 2 | `CustomAggregationStrategy` | `1-MINUTE-…-EXTERNAL` | `CustomBarAggregator` (this notebook) | OHLCV + derived features CSV |

**Single entry point: `aggregate_for_timeframe(timeframe, df)`** — pass any timeframe
string and it runs *both* aggregators against the same 1-min input, writing two CSVs:
`<base>_<month>_<tf>_nautilus.csv` and `<base>_<month>_<tf>_custom.csv`. No Parquet.

**Input (full month):** `C:/Users/HP/Desktop/MS/Dataset/FX_yyyy/Fx/EUROUSD/2024/04/`
**Outputs:** `ipynb/csv/*.csv`

Nautilus's `BarAggregation` enum natively supports `MINUTE / HOUR / DAY / WEEK / MONTH`
but **not `YEAR`** — so for `yearly` (and any other unit Nautilus rejects) the function
skips the Nautilus path and still produces the custom CSV. Whatever Nautilus accepts, it
runs; everything else is graceful no-op for that side.

In [10]:
import csv
import re
import sys
from dataclasses import dataclass, asdict, fields
from decimal import Decimal
from pathlib import Path

import numpy as np
import pandas as pd

# Resolve the project root robustly. Three search strategies, in order:
#   1. Walk UP from Path.cwd() looking for `core/instrument_factory.py`.
#   2. Walk UP from the notebook's own location.
#   3. Scan immediate children of cwd (VS Code may open an outer workspace).
def _find_project_root() -> Path:
    marker = Path("core") / "instrument_factory.py"

    def _walk_up(p: Path) -> Path | None:
        for parent in (p, *p.parents):
            if (parent / marker).is_file():
                return parent
        return None

    found = _walk_up(Path.cwd())
    if found:
        return found
    for var in ("__vsc_ipynb_file__", "__session__", "__file__"):
        nb = globals().get(var)
        if nb:
            found = _walk_up(Path(nb).resolve().parent)
            if found:
                return found
    for child in Path.cwd().iterdir():
        if child.is_dir() and (child / marker).is_file():
            return child
    raise RuntimeError(
        f"could not find project root (no `{marker}`) from cwd={Path.cwd()}"
    )


PROJECT_ROOT = _find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Single-day input retained for the 5-min Nautilus-vs-custom comparison demo below.
INPUT_CSV = r"C:/Users/HP/Desktop/MS/Dataset/FX_yyyy/Fx/EUROUSD/2015/01/02/02.01.2015_ASK_OHLCV.csv"
OUTPUT_NAUTILUS = PROJECT_ROOT / "eurusd_5min_nautilus.csv"
OUTPUT_CUSTOM = PROJECT_ROOT / "eurusd_5min_custom_features.csv"

# Full-month input + output directory for the multi-timeframe run.
INPUT_MONTH_DIR = Path(r"C:/Users/HP/Desktop/MS/Dataset/FX_yyyy/Fx/EUROUSD/2024/04")
MONTH_TAG = "2024_04"
IPYNB_DIR = PROJECT_ROOT / "ipynb"
CSV_OUT_DIR = IPYNB_DIR / "csv"
CSV_OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"cwd          : {Path.cwd()}")
print(f"project root : {PROJECT_ROOT}")
print(f"month input  : {INPUT_MONTH_DIR}")
print(f"csv out      : {CSV_OUT_DIR}")

BASE, QUOTE, VENUE_NAME, PRICE_TYPE = "EUR", "USD", "FOREX_MS", "ASK"
OUTPUT_TZ = "Asia/Kolkata"  # render output timestamps in source-file local time (IST, +05:30)

cwd          : d:\mcube_test_version_1_updated\m-cube_version1\ipynb
project root : d:\mcube_test_version_1_updated\m-cube_version1
month input  : C:\Users\HP\Desktop\MS\Dataset\FX_yyyy\Fx\EUROUSD\2024\04
csv out      : d:\mcube_test_version_1_updated\m-cube_version1\ipynb\csv


In [11]:
from nautilus_trader.backtest.engine import BacktestEngine
from nautilus_trader.config import BacktestEngineConfig, LoggingConfig, RiskEngineConfig, StrategyConfig
from nautilus_trader.model import TraderId
from nautilus_trader.model.currencies import USD
from nautilus_trader.model.data import Bar, BarType
from nautilus_trader.model.enums import AccountType, OmsType
from nautilus_trader.model.identifiers import InstrumentId
from nautilus_trader.model.objects import Money
from nautilus_trader.persistence.wranglers import BarDataWrangler
from nautilus_trader.trading.strategy import Strategy

from core.instrument_factory import create_instrument

In [12]:
def load_1min_csv(path: str) -> pd.DataFrame:
    """Read the FX 1-min OHLCV CSV → UTC-indexed DataFrame.

    Source format: `DD.MM.YYYY HH:MM:SS.fff GMT+HHMM`.
    """
    df = pd.read_csv(path)
    df["timestamp"] = pd.to_datetime(
        df["timestamp"], format="%d.%m.%Y %H:%M:%S.%f GMT%z", utc=True
    )
    df = df.set_index("timestamp").sort_index()
    return df[["open", "high", "low", "close", "volume"]].astype(float)


def load_1min_month(month_dir: str | Path, price_type: str = PRICE_TYPE) -> pd.DataFrame:
    """Load every daily `*_<price>_OHLCV.csv` under a month directory and concatenate.

    Expected layout: `<month_dir>/<DD>/<DD.MM.YYYY>_<price>_OHLCV.csv`.
    Returns a UTC-indexed, deduplicated, sorted DataFrame covering the whole month.
    """
    month_dir = Path(month_dir)
    pattern = f"*/*_{price_type}_OHLCV.csv"
    csvs = sorted(month_dir.glob(pattern))
    if not csvs:
        raise FileNotFoundError(f"no '{pattern}' files under {month_dir}")
    parts = [load_1min_csv(str(p)) for p in csvs]
    df = pd.concat(parts).sort_index()
    df = df[~df.index.duplicated(keep="first")]
    return df


df_1min = load_1min_csv(INPUT_CSV)
print(f"loaded {len(df_1min):,} 1-min bars from single-day demo file")
print(f"range: {df_1min.index.min()} → {df_1min.index.max()}")
df_1min.head()

loaded 1,230 1-min bars from single-day demo file
range: 2015-01-01 22:00:00+00:00 → 2015-01-02 18:29:00+00:00


,open,high,low,close,volume
timestamp,,,,,
2015-01-01 22:00:00+00:00,1.21066,1.21067,1.21059,1.21059,4.50
2015-01-01 22:01:00+00:00,1.21060,1.21067,1.21058,1.21066,15.75
2015-01-01 22:02:00+00:00,1.21061,1.21061,1.21058,1.21058,7.53
2015-01-01 22:03:00+00:00,1.21055,1.21065,1.21049,1.21063,13.53
2015-01-01 22:04:00+00:00,1.21064,1.21071,1.21050,1.21055,44.50


## Custom data class — `BarFeatures`

What `CustomBarAggregator` emits at the end of each window (regardless of timeframe).
Plain OHLCV columns plus derived features:

- **Geometry:** `range`, `body`, `upper_wick`, `lower_wick`
- **Returns:** `log_return`, `pct_return` (vs. previous bar's close)
- **Volume-price:** `typical_price`, `vwap` (cumulative session VWAP)
- **Volatility:** `tr` (true range), `atr_14` (Wilder-style EWM)
- **Trend:** `ema_9`, `ema_21`
- **Momentum:** `rsi_14`
- **Bookkeeping:** `n_ticks` (count of 1-min bars folded into this aggregated bar)

The lookback periods (9, 21, 14) are measured **in bars**, so they scale automatically
with whatever target timeframe you pick.

In [13]:
@dataclass
class BarFeatures:
    """Aggregated bar with derived features. Timeframe-agnostic."""

    timestamp: pd.Timestamp
    open: float
    high: float
    low: float
    close: float
    volume: float
    n_ticks: int
    range: float
    body: float
    upper_wick: float
    lower_wick: float
    log_return: float | None
    pct_return: float | None
    typical_price: float
    vwap: float
    tr: float
    atr_14: float
    ema_9: float
    ema_21: float
    rsi_14: float | None

    @classmethod
    def field_names(cls) -> list[str]:
        return [f.name for f in fields(cls)]

    def to_row(self) -> dict:
        return asdict(self)


# Back-compat alias for any downstream code that still imports the old name.
Bar5MinFeatures = BarFeatures

## Custom aggregator — `CustomBarAggregator(timeframe=...)`

Same buffer-and-fold strategy as before, generalized to any timeframe. Holds 1-min
bars in a buffer until the bucket boundary is crossed, then folds the buffer into a
single OHLCV bar and computes the derived features. Cross-window state (EMAs, ATR,
RSI, cumulative VWAP) is updated incrementally so each emit is O(1).

**Bucketing convention is left-open `(prev_close, close]`** to match Nautilus' default —
a 1-min bar timestamped exactly on a bucket boundary closes the prior window.

The bucket-close function `_bucket_close(ts, n, unit)` is the only thing that needs
to understand the calendar:

| unit | step | bucket-close logic |
|------|------|--------------------|
| `min` | n | `ts.ceil(f"{n}min")` |
| `h`   | n | `ts.ceil(f"{n}h")` |
| `d`   | n | `ts.ceil(f"{n}D")` |
| `w`   | n | round up to the next `n`-week boundary, anchored at Mon Jan 5 1970 |
| `mo`  | n | round up to the next `n`-month-start, anchored at Jan 1970 |
| `y`   | n | round up to the next `n`-year-start, anchored at Jan 1970 |

`parse_timeframe("45min")` → `(45, "min")`; `parse_timeframe("2w")` → `(2, "w")`;
`parse_timeframe("yearly")` → `(1, "y")`; etc.

In [14]:
# --- Timeframe parsing ---------------------------------------------------------

_TF_ALIAS = {
    "minutely": (1, "min"), "minute": (1, "min"),
    "hourly":   (1, "h"),   "hour":   (1, "h"),
    "daily":    (1, "d"),   "day":    (1, "d"),
    "weekly":   (1, "w"),   "week":   (1, "w"),
    "monthly":  (1, "mo"),  "month":  (1, "mo"),
    "yearly":   (1, "y"),   "year":   (1, "y"), "annual": (1, "y"),
}
# Suffix → canonical unit. 'm' on its own is treated as minute (intraday convention);
# month must be spelled 'mo'/'month'/'months' so it never collides.
_UNIT_NORMALIZE = {
    "m": "min", "min": "min", "mins": "min", "minute": "min", "minutes": "min",
    "h": "h", "hr": "h", "hrs": "h", "hour": "h", "hours": "h",
    "d": "d", "day": "d", "days": "d",
    "w": "w", "wk": "w", "wks": "w", "week": "w", "weeks": "w",
    "mo": "mo", "mos": "mo", "month": "mo", "months": "mo",
    "y": "y", "yr": "y", "yrs": "y", "year": "y", "years": "y",
}
_TF_PATTERN = re.compile(r"^\s*(\d+)?\s*([a-zA-Z]+)\s*$")


def parse_timeframe(tf: str) -> tuple[int, str]:
    """Parse '15min' / '1h' / '2w' / '1mo' / 'yearly' → (n, canonical_unit)."""
    s = tf.strip().lower().replace(" ", "")
    if s in _TF_ALIAS:
        return _TF_ALIAS[s]
    m = _TF_PATTERN.match(s)
    if not m:
        raise ValueError(f"could not parse timeframe: {tf!r}")
    n = int(m.group(1)) if m.group(1) else 1
    unit_raw = m.group(2)
    if unit_raw not in _UNIT_NORMALIZE:
        raise ValueError(f"unsupported unit {unit_raw!r} in timeframe {tf!r}")
    if n <= 0:
        raise ValueError(f"timeframe multiplier must be positive: {tf!r}")
    return n, _UNIT_NORMALIZE[unit_raw]


def normalize_timeframe(tf: str) -> str:
    """Canonical short form used for filenames: '15min', '1h', '2w', '1mo', '1y'."""
    n, unit = parse_timeframe(tf)
    return f"{n}{unit}"


# --- Bucket-close calculation --------------------------------------------------

# Anchor points (timezone-naive). Bucket boundaries are placed at:
#   week  → Mon Jan 5 1970 00:00 + k * (7n days)
#   month → Jan 1970 + k * n months
#   year  → Jan 1970 + k * n years
_WEEK_ANCHOR_NAIVE = pd.Timestamp("1970-01-05")


def _bucket_close(ts: pd.Timestamp, n: int, unit: str) -> pd.Timestamp:
    """Smallest bucket boundary >= ts (left-open: on-boundary ts closes prior window)."""
    if unit == "min":
        return ts.ceil(f"{n}min")
    if unit == "h":
        return ts.ceil(f"{n}h")
    if unit == "d":
        return ts.ceil(f"{n}D")

    tz = ts.tz
    if unit == "w":
        anchor = _WEEK_ANCHOR_NAIVE.tz_localize(tz) if tz is not None else _WEEK_ANCHOR_NAIVE
        secs_per = 7 * 86400 * n
        delta = (ts - anchor).total_seconds()
        k = int(delta // secs_per)
        if delta == k * secs_per:
            return anchor + pd.Timedelta(seconds=k * secs_per)
        return anchor + pd.Timedelta(seconds=(k + 1) * secs_per)

    if unit == "mo":
        month_idx = (ts.year - 1970) * 12 + (ts.month - 1)
        ns = getattr(ts, "nanosecond", 0) or 0
        on_boundary = (
            ts.day == 1 and ts.hour == 0 and ts.minute == 0
            and ts.second == 0 and ts.microsecond == 0 and ns == 0
        )
        if on_boundary and month_idx % n == 0:
            return ts
        group = month_idx // n
        close_idx = (group + 1) * n
        close_year, close_m0 = divmod(close_idx, 12)
        return pd.Timestamp(year=1970 + close_year, month=close_m0 + 1, day=1, tz=tz)

    if unit == "y":
        year_idx = ts.year - 1970
        ns = getattr(ts, "nanosecond", 0) or 0
        on_boundary = (
            ts.month == 1 and ts.day == 1 and ts.hour == 0 and ts.minute == 0
            and ts.second == 0 and ts.microsecond == 0 and ns == 0
        )
        if on_boundary and year_idx % n == 0:
            return ts
        group = year_idx // n
        return pd.Timestamp(year=1970 + (group + 1) * n, month=1, day=1, tz=tz)

    raise ValueError(f"unsupported unit: {unit!r}")


# --- Aggregator ---------------------------------------------------------------

class CustomBarAggregator:
    """Buffer 1-min bars, emit a `BarFeatures` whenever the bucket boundary is crossed.

    `timeframe` accepts any string `parse_timeframe` understands (e.g. '5min',
    '15min', '1h', '1d', '2w', '1mo', 'yearly').
    """

    def __init__(
        self,
        timeframe: str = "5min",
        ema_short_period: int = 9,
        ema_long_period: int = 21,
        rsi_period: int = 14,
        atr_period: int = 14,
    ) -> None:
        self.timeframe = timeframe
        self._n, self._unit = parse_timeframe(timeframe)
        self._alpha_ema_s = 2.0 / (ema_short_period + 1)
        self._alpha_ema_l = 2.0 / (ema_long_period + 1)
        self._alpha_rsi = 1.0 / rsi_period
        self._alpha_atr = 1.0 / atr_period

        # rolling per-window state
        self._buffer: list = []
        self._window_close: pd.Timestamp | None = None

        # cross-window state
        self._prev_close: float | None = None
        self._ema_s: float | None = None
        self._ema_l: float | None = None
        self._avg_gain: float = 0.0
        self._avg_loss: float = 0.0
        self._atr: float | None = None
        self._cum_pv: float = 0.0
        self._cum_v: float = 0.0

    @staticmethod
    def _bar_ts(bar) -> pd.Timestamp:
        return pd.Timestamp(bar.ts_event, unit="ns", tz="UTC")

    def on_bar(self, bar) -> BarFeatures | None:
        ts = self._bar_ts(bar)
        bucket_close = _bucket_close(ts, self._n, self._unit)

        emitted: BarFeatures | None = None
        if self._window_close is None:
            self._window_close = bucket_close
        elif bucket_close != self._window_close:
            emitted = self._close_window()
            self._window_close = bucket_close

        self._buffer.append(bar)
        return emitted

    def flush(self) -> BarFeatures | None:
        """Emit the trailing partial window (call from `on_stop`)."""
        if self._buffer:
            return self._close_window()
        return None

    def _close_window(self) -> BarFeatures:
        bars = self._buffer
        opens = float(bars[0].open)
        highs = max(float(b.high) for b in bars)
        lows = min(float(b.low) for b in bars)
        closes = float(bars[-1].close)
        volume = sum(float(b.volume) for b in bars)
        n_ticks = len(bars)

        # geometry
        rng = highs - lows
        body = abs(closes - opens)
        upper_wick = highs - max(opens, closes)
        lower_wick = min(opens, closes) - lows

        # returns vs. previous bar's close
        if self._prev_close is None or self._prev_close == 0:
            log_ret = None
            pct_ret = None
        else:
            log_ret = float(np.log(closes / self._prev_close))
            pct_ret = closes / self._prev_close - 1.0

        # cumulative VWAP
        typical = (highs + lows + closes) / 3.0
        self._cum_pv += typical * volume
        self._cum_v += volume
        vwap = self._cum_pv / self._cum_v if self._cum_v > 0 else typical

        # true range + Wilder ATR
        if self._prev_close is None:
            tr = rng
        else:
            tr = max(rng, abs(highs - self._prev_close), abs(lows - self._prev_close))
        self._atr = tr if self._atr is None else self._atr + self._alpha_atr * (tr - self._atr)

        # EMAs
        self._ema_s = closes if self._ema_s is None else self._ema_s + self._alpha_ema_s * (closes - self._ema_s)
        self._ema_l = closes if self._ema_l is None else self._ema_l + self._alpha_ema_l * (closes - self._ema_l)

        # RSI(14) — Wilder smoothing of gains/losses
        rsi: float | None = None
        if self._prev_close is not None:
            change = closes - self._prev_close
            gain = max(change, 0.0)
            loss = max(-change, 0.0)
            self._avg_gain += self._alpha_rsi * (gain - self._avg_gain)
            self._avg_loss += self._alpha_rsi * (loss - self._avg_loss)
            if self._avg_loss > 0:
                rs = self._avg_gain / self._avg_loss
                rsi = 100.0 - 100.0 / (1.0 + rs)
            elif self._avg_gain > 0:
                rsi = 100.0

        bar_out = BarFeatures(
            timestamp=self._window_close,
            open=opens,
            high=highs,
            low=lows,
            close=closes,
            volume=volume,
            n_ticks=n_ticks,
            range=rng,
            body=body,
            upper_wick=upper_wick,
            lower_wick=lower_wick,
            log_return=log_ret,
            pct_return=pct_ret,
            typical_price=typical,
            vwap=vwap,
            tr=tr,
            atr_14=self._atr,
            ema_9=self._ema_s,
            ema_21=self._ema_l,
            rsi_14=rsi,
        )

        self._prev_close = closes
        self._buffer = []
        return bar_out


# Back-compat alias so the original 5-min comparison cells below keep working.
FiveMinAggregator = CustomBarAggregator

## Strategy 1 — `NautilusInternalStrategy`

Subscribes to a **composite bar type** of the form
`<id>-5-MINUTE-<price>-INTERNAL@1-MINUTE-EXTERNAL`. The Nautilus `DataEngine`
sees the `INTERNAL@…-EXTERNAL` tail, spins up a `TimeBarAggregator` internally,
and feeds it from the 1-min EXTERNAL bars sitting in the engine's data buffer.
By the time `on_bar` fires, the bar passed in is already a 5-min bar. Strategy
logic = empty; we just append each bar to the output CSV.

In [15]:
class NautilusInternalConfig(StrategyConfig, frozen=True):
    instrument_id: InstrumentId
    bar_type: BarType  # the 5-min INTERNAL@1-MIN-EXTERNAL composite
    output_csv: str


class NautilusInternalStrategy(Strategy):
    """Subscribes to 5-min INTERNAL bars; writes each `on_bar` to CSV."""

    def __init__(self, config: NautilusInternalConfig) -> None:
        super().__init__(config)
        self._fh = None
        self._writer: csv.DictWriter | None = None
        self._n_bars = 0

    def on_start(self) -> None:
        self._fh = open(self.config.output_csv, "w", newline="", encoding="utf-8")
        self._writer = csv.DictWriter(
            self._fh, fieldnames=["timestamp", "open", "high", "low", "close", "volume"]
        )
        self._writer.writeheader()
        self.subscribe_bars(self.config.bar_type)
        self.log.info(f"subscribed to {self.config.bar_type}")

    def on_bar(self, bar: Bar) -> None:
        # bar is already a 5-min bar — Nautilus aggregated it for us
        self._writer.writerow({
            "timestamp": pd.Timestamp(bar.ts_event, unit="ns", tz="UTC")
                           .tz_convert(OUTPUT_TZ).isoformat(),
            "open": float(bar.open),
            "high": float(bar.high),
            "low": float(bar.low),
            "close": float(bar.close),
            "volume": float(bar.volume),
        })
        self._n_bars += 1

    def on_stop(self) -> None:
        if self._fh is not None:
            self._fh.close()
        self.log.info(f"wrote {self._n_bars} bars → {self.config.output_csv}")

## Strategy 2 — `CustomAggregationStrategy`

Subscribes to plain `1-MINUTE-…-EXTERNAL` bars. Each `on_bar` is a 1-min bar; the
strategy pushes it into the `FiveMinAggregator`. When the aggregator emits a
`Bar5MinFeatures` (i.e. the previous 5-min window has just closed), the strategy
writes that record to CSV. `on_stop` flushes the trailing partial window.

In [16]:
class CustomAggregationConfig(StrategyConfig, frozen=True):
    instrument_id: InstrumentId
    bar_type: BarType        # the 1-min EXTERNAL bar type
    output_csv: str
    timeframe: str = "5min"  # target aggregation timeframe — anything parse_timeframe accepts


class CustomAggregationStrategy(Strategy):
    """Subscribes to 1-min bars; runs them through `CustomBarAggregator(timeframe=...)`."""

    def __init__(self, config: CustomAggregationConfig) -> None:
        super().__init__(config)
        self._aggregator = CustomBarAggregator(timeframe=config.timeframe)
        self._fh = None
        self._writer: csv.DictWriter | None = None
        self._n_bars = 0

    def on_start(self) -> None:
        self._fh = open(self.config.output_csv, "w", newline="", encoding="utf-8")
        self._writer = csv.DictWriter(self._fh, fieldnames=BarFeatures.field_names())
        self._writer.writeheader()
        self.subscribe_bars(self.config.bar_type)
        self.log.info(f"subscribed to {self.config.bar_type} | tf={self.config.timeframe}")

    def _write(self, b: BarFeatures) -> None:
        row = b.to_row()
        row["timestamp"] = b.timestamp.tz_convert(OUTPUT_TZ).isoformat()
        self._writer.writerow(row)
        self._n_bars += 1

    def on_bar(self, bar: Bar) -> None:
        emitted = self._aggregator.on_bar(bar)
        if emitted is not None:
            self._write(emitted)

    def on_stop(self) -> None:
        trailing = self._aggregator.flush()
        if trailing is not None:
            self._write(trailing)
        if self._fh is not None:
            self._fh.close()
        self.log.info(f"wrote {self._n_bars} bars → {self.config.output_csv}")

## Backtest harness

Both strategies need the same input — 1-minute EXTERNAL bars in the engine's data
buffer. Strategy 1 then asks the engine for 5-min INTERNAL bars (which the engine
synthesises from the 1-min ones); Strategy 2 just consumes the 1-min stream
directly. The helper below wires up a fresh `BacktestEngine` for one run.

In [17]:
def _build_1min_bars(df: pd.DataFrame, instrument, price_type: str = PRICE_TYPE) -> tuple[BarType, list[Bar]]:
    bt_1min = BarType.from_str(f"{instrument.id}-1-MINUTE-{price_type}-EXTERNAL")
    bars = BarDataWrangler(bt_1min, instrument).process(df)
    return bt_1min, bars


def run_strategy(strategy_factory, df_1min_input: pd.DataFrame | None = None) -> None:
    """Spin up a BacktestEngine, load 1-min bars, attach a strategy via the factory.

    `strategy_factory(instrument, bar_type_1min)` must return a `Strategy` instance.
    `df_1min_input` defaults to the notebook-level `df_1min` (single-day demo).
    """
    instrument = create_instrument(BASE, QUOTE, venue=VENUE_NAME)
    src_df = df_1min if df_1min_input is None else df_1min_input
    bt_1min, bars = _build_1min_bars(src_df, instrument)

    engine = BacktestEngine(
        config=BacktestEngineConfig(
            trader_id=TraderId("AGG-DEMO-001"),
            logging=LoggingConfig(bypass_logging=True),
            risk_engine=RiskEngineConfig(bypass=True),
            run_analysis=False,
        )
    )
    engine.add_venue(
        venue=instrument.id.venue,
        oms_type=OmsType.NETTING,
        account_type=AccountType.MARGIN,
        starting_balances=[Money(1_000_000, USD)],
        base_currency=USD,
        default_leverage=Decimal(1),
    )
    engine.add_instrument(instrument)
    engine.add_data(bars)

    strategy = strategy_factory(instrument, bt_1min)
    engine.add_strategy(strategy)
    engine.run()
    engine.dispose()

### Run Strategy 1 — Nautilus internal aggregation

In [9]:
def make_internal_strategy(instrument, bt_1min: BarType) -> NautilusInternalStrategy:
    bt_5min_internal = BarType.from_str(
        f"{instrument.id}-5-MINUTE-{PRICE_TYPE}-INTERNAL@1-MINUTE-EXTERNAL"
    )
    cfg = NautilusInternalConfig(
        instrument_id=instrument.id,
        bar_type=bt_5min_internal,
        output_csv=str(OUTPUT_NAUTILUS),
    )
    return NautilusInternalStrategy(cfg)


run_strategy(make_internal_strategy)
df_5min_nautilus = pd.read_csv(OUTPUT_NAUTILUS)
print(f"nautilus internal: {len(df_5min_nautilus)} rows → {OUTPUT_NAUTILUS}")
df_5min_nautilus.head()

nautilus internal: 246 rows → d:\mcube_test_version_1_updated\m-cube_version1\eurusd_5min_nautilus.csv


,timestamp,open,high,low,close,volume
0,2015-01-02T03:30:00+05:30,1.21066,1.21067,1.21059,1.21059,5.0
1,2015-01-02T03:35:00+05:30,1.21060,1.21071,1.21045,1.21046,99.0
2,2015-01-02T03:40:00+05:30,1.21045,1.21047,1.21033,1.21038,37.0
3,2015-01-02T03:45:00+05:30,1.21037,1.21082,1.21029,1.21050,150.0
4,2015-01-02T03:50:00+05:30,1.21050,1.21057,1.21008,1.21011,137.0


### Run Strategy 2 — Custom aggregator + derived features

In [10]:
def make_custom_strategy(instrument, bt_1min: BarType) -> CustomAggregationStrategy:
    cfg = CustomAggregationConfig(
        instrument_id=instrument.id,
        bar_type=bt_1min,
        output_csv=str(OUTPUT_CUSTOM),
        timeframe="5min",
    )
    return CustomAggregationStrategy(cfg)


run_strategy(make_custom_strategy)
df_5min_custom = pd.read_csv(OUTPUT_CUSTOM)
print(f"custom aggregator (5min, single day): {len(df_5min_custom)} rows → {OUTPUT_CUSTOM}")
df_5min_custom.head()

custom aggregator (5min, single day): 247 rows → d:\mcube_test_version_1_updated\m-cube_version1\eurusd_5min_custom_features.csv


,timestamp,open,high,low,close,volume,n_ticks,range,body,upper_wick,lower_wick,log_return,pct_return,typical_price,vwap,tr,atr_14,ema_9,ema_21,rsi_14
0,2015-01-02T03:30:00+05:30,1.21066,1.21067,1.21059,1.21059,5.0,1,0.00008,0.00007,0.00001,0.00000,NaN,NaN,1.210617,1.210617,0.00008,0.000080,1.210590,1.210590,NaN
1,2015-01-02T03:35:00+05:30,1.21060,1.21071,1.21045,1.21046,99.0,5,0.00026,0.00014,0.00011,0.00001,-0.000107,-0.000107,1.210540,1.210544,0.00026,0.000093,1.210564,1.210578,0.000000
2,2015-01-02T03:40:00+05:30,1.21045,1.21047,1.21033,1.21038,37.0,5,0.00014,0.00007,0.00002,0.00005,-0.000066,-0.000066,1.210393,1.210504,0.00014,0.000096,1.210527,1.210560,0.000000
3,2015-01-02T03:45:00+05:30,1.21037,1.21082,1.21029,1.21050,150.0,5,0.00053,0.00013,0.00032,0.00008,0.000099,0.000099,1.210537,1.210521,0.00053,0.000127,1.210522,1.210555,39.167361
4,2015-01-02T03:50:00+05:30,1.21050,1.21057,1.21008,1.21011,137.0,5,0.00049,0.00039,0.00007,0.00003,-0.000322,-0.000322,1.210253,1.210435,0.00049,0.000153,1.210439,1.210514,16.520334


### Sanity check

On overlapping timestamps the OHLCV columns from the two paths should be
bit-identical — the only material difference is that the custom strategy adds
the trailing partial window (flushed in `on_stop`), whereas the Nautilus engine
stops emitting once the source data is exhausted.

In [11]:
a = df_5min_nautilus.set_index("timestamp")[["open", "high", "low", "close", "volume"]]
b = df_5min_custom.set_index("timestamp")[["open", "high", "low", "close", "volume"]]
common = a.index.intersection(b.index)
print(f"nautilus rows: {len(a)}   custom rows: {len(b)}   overlap: {len(common)}")
if len(common):
    print("max abs diff on overlap:")
    print((a.loc[common] - b.loc[common]).abs().max())

nautilus rows: 246   custom rows: 247   overlap: 246
max abs diff on overlap:
open      0.0
high      0.0
low       0.0
close     0.0
volume    0.0
dtype: float64


---

## Multi-timeframe run on the full month — `2024/04` EURUSD

Below: load every daily 1-min CSV under `INPUT_MONTH_DIR`, clip to UTC April, then
call `aggregate_for_timeframe(tf, df)` for each timeframe in `TIMEFRAMES`. Each call
runs **both** aggregators (Nautilus internal *and* the custom one) on the same 1-min
input and writes two CSVs into `ipynb/csv/`. No Parquet.

In [18]:
df_1min_month = load_1min_month(INPUT_MONTH_DIR)

# Source files are named by IST date (e.g. 01.04.2024_ASK_OHLCV.csv), so the
# first file contains a few hours of bars that — after converting to UTC — fall
# on March 31. Clip strictly to UTC April 2024 so nothing from March 31 (or
# May 1 spilled in from the 30.04 file) appears in any output.
_MONTH_START = pd.Timestamp("2024-04-01", tz="UTC")
_MONTH_END = pd.Timestamp("2024-05-01", tz="UTC")
_before = len(df_1min_month)
df_1min_month = df_1min_month.loc[
    (df_1min_month.index >= _MONTH_START) & (df_1min_month.index < _MONTH_END)
]
print(f"loaded {_before:,} 1-min bars; kept {len(df_1min_month):,} in UTC April 2024 "
      f"(dropped {_before - len(df_1min_month):,})")
print(f"range: {df_1min_month.index.min()} → {df_1min_month.index.max()}")
df_1min_month.head()

loaded 31,530 1-min bars; kept 31,350 in UTC April 2024 (dropped 180)
range: 2024-04-01 00:00:00+00:00 → 2024-04-30 18:29:00+00:00


,open,high,low,close,volume
timestamp,,,,,
2024-04-01 00:00:00+00:00,1.07934,1.07937,1.07921,1.07924,72.92
2024-04-01 00:01:00+00:00,1.07923,1.07949,1.07923,1.07936,135.36
2024-04-01 00:02:00+00:00,1.07937,1.07938,1.07933,1.07937,119.16
2024-04-01 00:03:00+00:00,1.07935,1.07956,1.07934,1.07949,121.65
2024-04-01 00:04:00+00:00,1.07948,1.07952,1.07947,1.07948,31.50


In [19]:
# Map our canonical aggregator units → Nautilus `BarAggregation` enum names.
# YEAR is intentionally absent — Nautilus has no native YEAR aggregation, so
# yearly bars come from the custom path only.
_NAUTILUS_UNIT = {
    "min": "MINUTE",
    "h":   "HOUR",
    "d":   "DAY",
    "w":   "WEEK",
    "mo":  "MONTH",
}


def aggregate_for_timeframe(
    timeframe: str,
    df_1min_input: pd.DataFrame,
    csv_dir: Path = CSV_OUT_DIR,
    base_name: str = "eurusd",
    month_tag: str = MONTH_TAG,
) -> dict[str, Path | None]:
    """Run **both** the Nautilus-internal aggregator and the custom aggregator on
    `df_1min_input` for the given `timeframe`. Writes up to two CSVs into `csv_dir`:

        <base>_<month_tag>_<tf>_nautilus.csv   (OHLCV from Nautilus DataEngine)
        <base>_<month_tag>_<tf>_custom.csv     (OHLCV + derived features from CustomBarAggregator)

    Returns `{"nautilus": Path | None, "custom": Path}`. The Nautilus entry is `None`
    when Nautilus does not natively support the timeframe's unit (e.g. yearly) or
    when the Nautilus run raises — the custom CSV is always produced.
    """
    n, unit = parse_timeframe(timeframe)
    tf_tag = normalize_timeframe(timeframe)
    custom_csv = csv_dir / f"{base_name}_{month_tag}_{tf_tag}_custom.csv"
    nautilus_csv = csv_dir / f"{base_name}_{month_tag}_{tf_tag}_nautilus.csv"

    # --- Custom path (always runs) ---
    def custom_factory(instrument, bt_1min: BarType) -> CustomAggregationStrategy:
        cfg = CustomAggregationConfig(
            instrument_id=instrument.id,
            bar_type=bt_1min,
            output_csv=str(custom_csv),
            timeframe=timeframe,
        )
        return CustomAggregationStrategy(cfg)

    run_strategy(custom_factory, df_1min_input=df_1min_input)

    # --- Nautilus internal path (skip if unit unsupported) ---
    nautilus_path: Path | None = None
    nautilus_unit = _NAUTILUS_UNIT.get(unit)
    if nautilus_unit is None:
        print(f"  [{timeframe}] nautilus: SKIPPED (unit '{unit}' not supported by Nautilus)")
    else:
        def nautilus_factory(instrument, bt_1min: BarType) -> NautilusInternalStrategy:
            bt = BarType.from_str(
                f"{instrument.id}-{n}-{nautilus_unit}-{PRICE_TYPE}-INTERNAL@1-MINUTE-EXTERNAL"
            )
            cfg = NautilusInternalConfig(
                instrument_id=instrument.id,
                bar_type=bt,
                output_csv=str(nautilus_csv),
            )
            return NautilusInternalStrategy(cfg)

        try:
            run_strategy(nautilus_factory, df_1min_input=df_1min_input)
            nautilus_path = nautilus_csv
        except Exception as exc:
            print(f"  [{timeframe}] nautilus: FAILED ({type(exc).__name__}: {exc}) — custom CSV still produced")

    return {"nautilus": nautilus_path, "custom": custom_csv}

In [20]:
TIMEFRAMES = [
    "5min", "15min", "30min", "45min", "60min",
    "1d",
    "1w", "2w",
    "1mo", "2mo",
    "yearly",
]

summary_rows = []
for tf in TIMEFRAMES:
    print(f"running {tf} ...")
    out = aggregate_for_timeframe(tf, df_1min_month)
    custom_rows = len(pd.read_csv(out["custom"]))
    nautilus_rows = len(pd.read_csv(out["nautilus"])) if out["nautilus"] else None
    summary_rows.append({
        "timeframe": tf,
        "canonical": normalize_timeframe(tf),
        "custom_rows": custom_rows,
        "nautilus_rows": nautilus_rows if nautilus_rows is not None else "—",
        "custom_csv": out["custom"].name,
        "nautilus_csv": out["nautilus"].name if out["nautilus"] else "—",
    })

summary = pd.DataFrame(summary_rows)
print(f"\nwrote CSVs → {CSV_OUT_DIR}")
summary

running 5min ...
running 15min ...
running 30min ...
running 45min ...
running 60min ...
running 1d ...
running 1w ...
running 2w ...
running 1mo ...
running 2mo ...
running yearly ...
  [yearly] nautilus: SKIPPED (unit 'y' not supported by Nautilus)

wrote CSVs → d:\mcube_test_version_1_updated\m-cube_version1\ipynb\csv


,timeframe,canonical,custom_rows,nautilus_rows,custom_csv,nautilus_csv
0,5min,5min,6275,8574,eurusd_2024_04_5min_custom.csv,eurusd_2024_04_5min_nautilus.csv
1,15min,15min,2095,2858,eurusd_2024_04_15min_custom.csv,eurusd_2024_04_15min_nautilus.csv
2,30min,30min,1050,1429,eurusd_2024_04_30min_custom.csv,eurusd_2024_04_30min_nautilus.csv
3,45min,45min,702,953,eurusd_2024_04_45min_custom.csv,eurusd_2024_04_45min_nautilus.csv
4,60min,60min,528,715,eurusd_2024_04_60min_custom.csv,eurusd_2024_04_60min_nautilus.csv
5,1d,1d,27,30,eurusd_2024_04_1d_custom.csv,eurusd_2024_04_1d_nautilus.csv
6,1w,1w,6,5,eurusd_2024_04_1w_custom.csv,eurusd_2024_04_1w_nautilus.csv
7,2w,2w,4,3,eurusd_2024_04_2w_custom.csv,eurusd_2024_04_2w_nautilus.csv
8,1mo,1mo,2,1,eurusd_2024_04_1mo_custom.csv,eurusd_2024_04_1mo_nautilus.csv
9,2mo,2mo,1,0,eurusd_2024_04_2mo_custom.csv,eurusd_2024_04_2mo_nautilus.csv
